# Tools  calling

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

<img src="Tool calling.png" alt="Step 1 is the analyze calling tool in this graph">

To make tools that you have defined available for use by a model, you must bind them using bind_tools. In subsequent invocations, the model can choose to call any of the bound tools as needed.
Some model providers offer built-in tools that can be enabled via model or invocation parameters (e.g. ChatOpenAI, ChatAnthropic). Check the respective provider reference for details.

In [1]:
# You can create model using this method also,
from langchain_google_genai import ChatGoogleGenerativeAI

# Option2 [More popular]
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [2]:
from langchain.tools import tool

# tool Decoration
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

# Bind tool with model
model_with_tools = model.bind_tools([get_weather])  



Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [6]:
tool_call

{'name': 'get_weather',
 'args': {'location': 'Boston'},
 'id': '13a55d57-8170-4047-8130-0ea2c4375d12',
 'type': 'tool_call'}

In [ ]:
# Notice this, if the question is related to 1 of the tool in AI , 
# it will return tool call reminder message, which you need to call in step2 to get result and parse back to model
# but if we ask the common question not involving the tool, model will answer right away
model_with_tools.invoke("What's the capital of Thailand?")

AIMessage(content='The capital of Thailand is Bangkok.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--c24e5a6f-bc12-4e0b-84c4-b51f0ae32f3c-0', usage_metadata={'input_tokens': 48, 'output_tokens': 7, 'total_tokens': 94})

When binding user-defined tools, the model’s response includes a request to execute a tool. When using a model separately from an agent, it is up to you to execute the requested tool and return the result back to the model for use in subsequent reasoning. When using an agent, the agent loop will handle the tool execution loop for you.

Below, we show some common ways you can use tool calling.

## Tool execution loop

When a model returns tool calls, you need to execute the tools and pass the results back to the model. This creates a conversation loop where the model can use tool results to generate its final response. LangChain includes agent abstractions that handle this orchestration for you.

Here’s a simple example of how to do this:

In [7]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


Step 1 return AI message analyze and telling itself that it has to call tool

- Note: as I mentioned above, if the <b>model decide to use a tool it will return the tool_call</b> ai message

But if it decides the <b>user prompt is not related to any tools, it will just return normal ai response message.</b>

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
original_messages = [item for item in messages]
print('message1',messages[0].keys() )
print('message1',messages)
print('')
ai_msg = model_with_tools.invoke(messages)
print('ai_msg',ai_msg)
print('')
messages.append(ai_msg)
print('message2',messages[0].keys() )
print('message2',messages)
print('')

message1 dict_keys(['role', 'content'])
message1 [{'role': 'user', 'content': "What's the weather in Boston?"}]

ai_msg content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--57915621-766e-4e04-8f6d-9179eb39a3d6-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '45f0d8dc-571e-4c8e-beb9-3a866ca5ff27', 'type': 'tool_call'}] usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 111}

message2 dict_keys(['role', 'content'])
message2 [{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--5791

In [29]:
vars(ai_msg)

{'content': '',
 'additional_kwargs': {'function_call': {'name': 'get_weather',
   'arguments': '{"location": "Boston"}'}},
 'response_metadata': {'prompt_feedback': {'block_reason': 0,
   'safety_ratings': []},
  'finish_reason': 'STOP',
  'safety_ratings': []},
 'type': 'ai',
 'name': None,
 'id': 'run--fd151552-7bdb-4cac-8cc1-e5d3cd17e43c-0',
 'example': False,
 'tool_calls': [{'name': 'get_weather',
   'args': {'location': 'Boston'},
   'id': '70e4bb78-40f0-4d6b-9f33-8980ab449a25',
   'type': 'tool_call'}],
 'invalid_tool_calls': [],
 'usage_metadata': {'input_tokens': 48,
  'output_tokens': 15,
  'total_tokens': 111}}

In [30]:
print(messages)

[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--fd151552-7bdb-4cac-8cc1-e5d3cd17e43c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '70e4bb78-40f0-4d6b-9f33-8980ab449a25', 'type': 'tool_call'}], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 111})]


In [12]:
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    print(tool_result)
    messages.append(tool_result)
print('====================')
print(messages)

content="It's sunny in Boston." name='get_weather' tool_call_id='235519ab-90c9-4ecf-998e-f0462819f22a'
[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--0ca1a2ff-58f4-4a0b-a5ff-de1405937ce5-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '235519ab-90c9-4ecf-998e-f0462819f22a', 'type': 'tool_call'}], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 111}), ToolMessage(content="It's sunny in Boston.", name='get_weather', tool_call_id='235519ab-90c9-4ecf-998e-f0462819f22a'), ToolMessage(content="It's sunny in Boston.", name='get_weather', tool_call_id='235519ab-90c9-4ecf-998e-f0462819f22a')]


In [13]:
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

<bound method BaseMessage.text of AIMessage(content="It's sunny in Boston.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--9cbacc7e-7630-495b-a7a0-79e6affb4e02-0', usage_metadata={'input_tokens': 104, 'output_tokens': 7, 'total_tokens': 111})>


Therefore, need to have code to check whether AIMessage from step 1 has tool_call tag or not

In [ ]:
# If AIMessage doesn't have tool call
if not ai_msg.tool_calls:
    print("--- Model returned a direct answer (no tools needed) ---")
    print(ai_msg.content)
    # The process ends here.

# If AI Message has tool call
else:
    print("--- Model requested tool execution ---")
    messages.append(ai_msg) # Append the model's plan

    # 2. Execute tools and collect results (Step 2)
    for tool_call in ai_msg.tool_calls:
        # ⚠️ This is where you would extract args and run the actual function
        
        # Simulating tool execution and result:
        if tool_call.name == "get_weather":
            tool_output_data = "The current weather in Boston is 72°F and sunny."
        else:
            tool_output_data = "Error: Unknown tool."

        # Create the ToolMessage and append the observation
        tool_result = ToolMessage(
            content=tool_output_data,
            tool_call_id=tool_call.id # Must link the result back to the request
        )
        messages.append(tool_result)

    # 3. Pass results back to model for final response (Step 3)
    final_response = model_with_tools.invoke(messages)
    print("--- Final Synthesized Response ---")
    print(final_response.content)

But The Agent AI has this embedded in their code and perform the decision and orchestration for us automatically 